In [1]:
#Import necessary libraries
import os
import glob
import pandas as pd
import numpy as np
import geopandas as gpd
import rioxarray as rxr
import xarray as xr
from climate_indices import compute
from climate_indices import indices
import matplotlib.pyplot as plt
import seaborn as sns




In [ ]:
#Paths 
CHIRPS_PATH = "chirps_africa"
import os
from pathlib import Path
BASE_DIR = os.getcwd() 
SHAPE_DIR = os.path.join(BASE_DIR, "shape_files")
ETH_SHAPE = os.path.join(SHAPE_DIR, "gadm41_ETH_0.shp")
OUTPUT_PATH = "processed_data"
os.makedirs(OUTPUT_PATH, exist_ok=True)


In [ ]:
#Load Ethiopia shapefile
# Load Ethiopia shapefile (with checks for companion files)
# For a shapefile to load correctly you need the .shp, .shx, .dbf and .prj files in the same folder.
required_files = [
    ETH_SHAPE,
    ETH_SHAPE.replace('.shp', '.shx'),
    ETH_SHAPE.replace('.shp', '.dbf'),
    ETH_SHAPE.replace('.shp', '.prj'),
]
missing = [f for f in required_files if not os.path.exists(f)]
if missing:
    # try GeoPackage fallback if present, else raise informative error
    gpkg_path = os.path.join(SHAPE_DIR, 'gadm41_ETH.gpkg')
    if os.path.exists(gpkg_path):
        eth_shape = gpd.read_file(gpkg_path, layer='gadm41_ETH_0').to_crs('EPSG:4326')
    else:
        raise FileNotFoundError(f'Missing shapefile components: {missing}')
    eth_shape = gpd.read_file(ETH_SHAPE).to_crs('EPSG:4326')


eth_shape = gpd.read_file(ETH_SHAPE).to_crs('EPSG:4326')

In [4]:
print("CHIRPS_PATH =", CHIRPS_PATH)

CHIRPS_PATH = /home/negasa/Development/chirps_africa


In [ ]:


# 1. Check .tif files
tif_files = sorted(glob.glob(os.path.join(CHIRPS_PATH, "*.tif")))
print(f"Found {len(tif_files)} CHIRPS files.")

if len(tif_files) == 0:
    raise RuntimeError("No .tif files found in the given CHIRPS_PATH. Check your directory or pattern.")

monthly_data = []
for tif in tif_files:
    print(f"Processing: {os.path.basename(tif)}")
    try:
        # Open as DataArray
        da = rxr.open_rasterio(tif, masked=True).squeeze()
        # Clip with the shapefile
        da_clip = da.rio.clip(eth_shape.geometry, eth_shape.crs, drop=True, invert=False)
        # If all values are nan (empty after clip), skip
        if da_clip.isnull().all():
            print(f"WARN: All values are NaN after clipping for {os.path.basename(tif)}. Skipping.")
            continue
        # Extract date from the filename
        parts = os.path.basename(tif).split(".")
        year = int(parts[2])
        month = int(parts[3])
        da_clip = da_clip.assign_coords(time=pd.Timestamp(year=year, month=month, day=1))
        monthly_data.append(da_clip)
    except Exception as e:
        print(f"ERROR processing {tif}: {e}")

print(f"Total clipped rasters for stacking: {len(monthly_data)}")
if not monthly_data:
    raise ValueError("monthly_data is empty after processing; check data sources and shapefile overlap.")


Found 536 CHIRPS files.
Processing: chirps-v3.0.1981.01.tif
Processing: chirps-v3.0.1981.02.tif
Processing: chirps-v3.0.1981.03.tif
Processing: chirps-v3.0.1981.04.tif
Processing: chirps-v3.0.1981.05.tif
Processing: chirps-v3.0.1981.06.tif
Processing: chirps-v3.0.1981.07.tif
Processing: chirps-v3.0.1981.08.tif
Processing: chirps-v3.0.1981.09.tif
Processing: chirps-v3.0.1981.10.tif
Processing: chirps-v3.0.1981.11.tif
Processing: chirps-v3.0.1981.12.tif
Processing: chirps-v3.0.1982.01.tif
Processing: chirps-v3.0.1982.02.tif
Processing: chirps-v3.0.1982.03.tif
Processing: chirps-v3.0.1982.04.tif
Processing: chirps-v3.0.1982.05.tif
Processing: chirps-v3.0.1982.06.tif
Processing: chirps-v3.0.1982.07.tif
Processing: chirps-v3.0.1982.08.tif
Processing: chirps-v3.0.1982.09.tif
Processing: chirps-v3.0.1982.10.tif
Processing: chirps-v3.0.1982.11.tif
Processing: chirps-v3.0.1982.12.tif
Processing: chirps-v3.0.1983.01.tif
Processing: chirps-v3.0.1983.02.tif
Processing: chirps-v3.0.1983.03.tif
Proc

In [6]:
# Open first raster just to check
sample_da = rxr.open_rasterio(tif_files[0], masked=True)
print("Raster CRS:", sample_da.rio.crs)
print("Shapefile CRS:", eth_shape.crs)

Raster CRS: EPSG:4326
Shapefile CRS: EPSG:4326


In [8]:
#load and clip CHIRPS monthly rasters
tif_files = sorted(glob.glob(os.path.join(CHIRPS_PATH, "*.tif")))
monthly_data = []
for tif in tif_files:
    print (f"Processing: ", os.path.basename(tif))
    #Opens GeoTIFF as a DataArray
    da = rxr.open_rasterio(tif, masked=True).squeeze()
    da= da.rio.clip(eth_shape.geometry, eth_shape.crs, drop=True, invert=False)
    
    #Extract date from the filename
    parts = os.path.basename(tif).split(".")
    year = int(parts[2])
    month = int(parts[3])
    da = da.assign_coords(time=pd.Timestamp(year=year, month=month, day=1))
    
    monthly_data.append(da)

Processing:  chirps-v3.0.1981.01.tif
Processing:  chirps-v3.0.1981.02.tif
Processing:  chirps-v3.0.1981.03.tif
Processing:  chirps-v3.0.1981.04.tif
Processing:  chirps-v3.0.1981.05.tif
Processing:  chirps-v3.0.1981.06.tif
Processing:  chirps-v3.0.1981.07.tif
Processing:  chirps-v3.0.1981.08.tif
Processing:  chirps-v3.0.1981.09.tif
Processing:  chirps-v3.0.1981.10.tif
Processing:  chirps-v3.0.1981.11.tif
Processing:  chirps-v3.0.1981.12.tif
Processing:  chirps-v3.0.1982.01.tif
Processing:  chirps-v3.0.1982.02.tif
Processing:  chirps-v3.0.1982.03.tif
Processing:  chirps-v3.0.1982.04.tif
Processing:  chirps-v3.0.1982.05.tif
Processing:  chirps-v3.0.1982.06.tif
Processing:  chirps-v3.0.1982.07.tif
Processing:  chirps-v3.0.1982.08.tif
Processing:  chirps-v3.0.1982.09.tif
Processing:  chirps-v3.0.1982.10.tif
Processing:  chirps-v3.0.1982.11.tif
Processing:  chirps-v3.0.1982.12.tif
Processing:  chirps-v3.0.1983.01.tif
Processing:  chirps-v3.0.1983.02.tif
Processing:  chirps-v3.0.1983.03.tif
P

In [9]:
#Stack into xarray dataset
rain = xr.concat(monthly_data, dim="time")
rain.name = "precipitation"
rain.attrs["units"] = "mm"
rain.attrs["long_name"] = "Monthly Precipitation"

#save clipped rainfall cube
rain.to_netcdf(os.path.join(OUTPUT_PATH, "ethiopia_monthly_rainfall.nc"))
print("Saved: ethiopia_monthly_rainfall.nc")


Saved: ethiopia_monthly_rainfall.nc


In [10]:
#Convert to area-average time series
rain_mean =  rain.mean (dim=["x","y"])  
rain_df = rain_mean.to_dataframe().reset_index()


In [12]:
#compute SPI
#prepare precipitation array (montly)
precip = rain_df['precipitation'].values.astype(np.float32)

In [13]:
#SPI Parameters
scale_3 = 3  # 3-month SPI
scale_6 = 6  # 6-month SPI
from climate_indices.compute import Periodicity
from climate_indices.indices import Distribution


spi3 = indices.spi(
    values = precip,
    scale = scale_3,
    distribution = Distribution.gamma,
    data_start_year = rain_df['time'].dt.year.iloc[0],
    calibration_year_initial =1981,
    calibration_year_final = 2010,
    periodicity=Periodicity.monthly
    
)

spi6 = indices.spi(
    values = precip,
    scale = scale_6,
    distribution = Distribution.gamma,
    data_start_year = rain_df['time'].dt.year.iloc[0],
    calibration_year_initial =1981,
    calibration_year_final = 2010,
    periodicity=Periodicity.monthly
)

#Add SPI to dataframe
rain_df['SPI_3'] = spi3
rain_df['SPI_6'] = spi6


In [14]:
#Save final ML ready dataset

rain_df.to_csv(os.path.join(OUTPUT_PATH, "ethiopia_rainfall_spi.csv"), index=False)
print("Saved: ethiopia_rainfall_spi.csv")
print ("Preprocessing completed successfully.")

Saved: ethiopia_rainfall_spi.csv
Preprocessing completed successfully.
